# Pipeline Bronze → Silver — ENEM 1998–2025

Este notebook é o ponto de execução operacional do pipeline. Ele processa os ZIPs armazenados na Bronze, seleciona as variáveis definidas no catálogo, filtra a UF desejada, grava Parquets na Silver, limpa o staging de forma segura e produz um manifesto final.

## Modelo de saída

- **1998–2023:** um Parquet anual de microdados selecionados;
- **2024–2025:** dois Parquets anuais independentes, `participantes` e `resultados`;

- **total esperado:** 30 Parquets para 28 edições.

> Em 2024–2025 não existe chave individual comum entre `NU_INSCRICAO` e `NU_SEQUENCIAL`. As bases não são unidas linha a linha.


## 1. Preparação

Execute o notebook na raiz do projeto ou dentro de `notebook`/`notebooks`.


In [21]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


diretorio_atual = Path.cwd().resolve()

if diretorio_atual.name in {"notebook", "notebooks"}:
    raiz_projeto = diretorio_atual.parent
else:
    raiz_projeto = diretorio_atual

src_dir = raiz_projeto / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"Raiz detectada: {raiz_projeto}")
print(f"Código-fonte  : {src_dir}")


Raiz detectada: /home/akel/PycharmProjects/ENEM
Código-fonte  : /home/akel/PycharmProjects/ENEM/src


In [22]:
from enem_pipeline.catalogo import carregar_catalogo
from enem_pipeline.config import (
    ANO_FINAL,
    ANO_INICIAL,
    BRONZE_DIR,
    CATALOGO_VARIAVEIS,
    CODIFICACAO_MICRODADOS_PADRAO,
    CODIFICACAO_MICRODADOS_POR_ANO,
    METADATA_DIR,
    SILVER_DIR,
    STAGING_DIR,
    UF_PADRAO,
)
from enem_pipeline.extracao import localizar_zip
from enem_pipeline.gravacao import (
    construir_caminho_parquet,
    construir_caminho_parquet_separado,
)
from enem_pipeline.processamento import processar_varios_anos


catalogo = carregar_catalogo()


## 2. Parâmetros de execução

As operações potencialmente longas ou destrutivas ficam desativadas por padrão.

- `EXECUTAR_PIPELINE`: processa os anos informados;
- `LIMPAR_STAGING_APOS_SUCESSO`: remove somente os CSVs principais depois que a Silver foi validada;
- `EXECUTAR_AUDITORIA_FINAL`: relê e valida todas as Silver;
- `SALVAR_MANIFESTO`: grava o manifesto em `data/metadata`.


In [23]:
UF_PROCESSAMENTO = UF_PADRAO

EXECUTAR_PIPELINE = False
LIMPAR_STAGING_APOS_SUCESSO = True
CONTINUAR_EM_ERRO = False

EXECUTAR_AUDITORIA_FINAL = False
SALVAR_MANIFESTO = False

print("UF                         :", UF_PROCESSAMENTO)
print("Executar pipeline          :", EXECUTAR_PIPELINE)
print("Limpar staging             :", LIMPAR_STAGING_APOS_SUCESSO)
print("Executar auditoria final   :", EXECUTAR_AUDITORIA_FINAL)
print("Salvar manifesto           :", SALVAR_MANIFESTO)


UF                         : PA
Executar pipeline          : False
Limpar staging             : True
Executar auditoria final   : False
Salvar manifesto           : False


## 3. Verificações preliminares

Esta etapa confirma o catálogo e os 28 ZIPs antes de iniciar qualquer extração.


In [24]:
assert BRONZE_DIR.exists(), f"Bronze não encontrada: {BRONZE_DIR}"
assert CATALOGO_VARIAVEIS.exists(), f"Catálogo não encontrado: {CATALOGO_VARIAVEIS}"
assert catalogo.shape[1] == 28

zips_encontrados = []

for ano in range(ANO_INICIAL, ANO_FINAL + 1):
    zips_encontrados.append(localizar_zip(ano))

assert len(zips_encontrados) == 28

print("Catálogo e 28 ZIPs validados.")


Catálogo e 28 ZIPs validados.


In [25]:
codificacoes = pd.DataFrame(
    {
        "ano": range(ANO_INICIAL, ANO_FINAL + 1),
        "codificacao": [
            CODIFICACAO_MICRODADOS_POR_ANO.get(
                ano,
                CODIFICACAO_MICRODADOS_PADRAO,
            )
            for ano in range(ANO_INICIAL, ANO_FINAL + 1)
        ],
    }
)

display(codificacoes.groupby("codificacao")["ano"].agg(list).to_frame())


,ano
codificacao,
latin-1,"[2006, 2008, 2011, 2012, 2013, 2014, 2015, 201..."
utf-8,"[1998, 1999, 2000, 2001, 2002, 2003, 2004, 200..."


## 4. Situação atual da Silver

Esta verificação consulta apenas a existência dos arquivos esperados; não lê os Parquets.


In [26]:
def destinos_esperados_ano(ano, uf):
    if ano <= 2023:
        return {
            "microdados": construir_caminho_parquet(ano, uf),
        }

    return {
        "participantes": construir_caminho_parquet_separado(
            ano, uf, "participantes"
        ),
        "resultados": construir_caminho_parquet_separado(
            ano, uf, "resultados"
        ),
    }


registros_situacao = []

for ano in range(ANO_INICIAL, ANO_FINAL + 1):
    for base, caminho in destinos_esperados_ano(ano, UF_PROCESSAMENTO).items():
        registros_situacao.append(
            {
                "ano": ano,
                "layout": "unico" if ano <= 2023 else "separado",
                "base": base,
                "existe": caminho.exists(),
                "tamanho_mb": caminho.stat().st_size / 1024**2 if caminho.exists() else None,
                "arquivo": caminho,
            }
        )

situacao_silver = pd.DataFrame(registros_situacao)

display(
    situacao_silver.groupby(["layout", "existe"])
    .size()
    .rename("arquivos")
    .to_frame()
)


,,arquivos
layout,existe,
separado,False,4
unico,False,26


In [27]:
anos_pendentes = sorted(
    situacao_silver.loc[~situacao_silver["existe"], "ano"].unique().tolist()
)

if anos_pendentes:
    print("Anos com Silver pendente:", anos_pendentes)
else:
    print("Todas as Silver esperadas já existem.")


Anos com Silver pendente: [1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


## 5. Definição dos lotes

Os lotes reduzem o impacto de uma eventual falha e deixam o progresso legível. Anos já processados são reutilizados e validados.


In [28]:
LOTES_PROCESSAMENTO = {
    "1998_2003": range(1998, 2004),
    "2004_2008": range(2004, 2009),
    "2009_2013": range(2009, 2014),
    "2014_2018": range(2014, 2019),
    "2019_2023": range(2019, 2024),
    "2024_2025": range(2024, 2026),
}

pd.DataFrame(
    {
        "lote": LOTES_PROCESSAMENTO.keys(),
        "anos": [list(anos) for anos in LOTES_PROCESSAMENTO.values()],
    }
)


,lote,anos
0,1998_2003,"[1998, 1999, 2000, 2001, 2002, 2003]"
1,2004_2008,"[2004, 2005, 2006, 2007, 2008]"
2,2009_2013,"[2009, 2010, 2011, 2012, 2013]"
3,2014_2018,"[2014, 2015, 2016, 2017, 2018]"
4,2019_2023,"[2019, 2020, 2021, 2022, 2023]"
5,2024_2025,"[2024, 2025]"


## 6. Execução Bronze → Silver

Para iniciar, altere `EXECUTAR_PIPELINE = True` na seção de parâmetros. Com a limpeza ativada, cada CSV do staging é removido somente após a existência e validação da Silver correspondente.


In [29]:
EXECUTAR_PIPELINE =True

In [30]:
resultados_lotes = {}

if EXECUTAR_PIPELINE:
    for nome_lote, anos in LOTES_PROCESSAMENTO.items():
        print(f"\n=== Lote {nome_lote} ===")

        resultados_lotes[nome_lote] = processar_varios_anos(
            anos=anos,
            uf=UF_PROCESSAMENTO,
            catalogo=catalogo,
            continuar_em_erro=CONTINUAR_EM_ERRO,
            limpar_staging_apos_sucesso=LIMPAR_STAGING_APOS_SUCESSO,
        )
else:
    print("Pipeline não executado. Altere EXECUTAR_PIPELINE para True quando desejar iniciar.")



=== Lote 1998_2003 ===
[1/6] Processando 1998...
  processado: 288 registros.
  staging: 60.14 MB liberados.
[2/6] Processando 1999...
  processado: 6,518 registros.
  staging: 138.05 MB liberados.
[3/6] Processando 2000...
  processado: 6,219 registros.
  staging: 165.59 MB liberados.
[4/6] Processando 2001...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  processado: 43,918 registros.
  staging: 1026.47 MB liberados.
[5/6] Processando 2002...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  processado: 36,407 registros.
  staging: 1014.89 MB liberados.
[6/6] Processando 2003...


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  processado: 248,061 participantes | 248,061 resultados.
  staging: 2046.08 MB liberados.
[2/2] Processando 2025...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  processado: 289,328 participantes | 289,328 resultados.
  staging: 2506.84 MB liberados.


In [31]:
if resultados_lotes:
    resumo_execucao = pd.concat(
        resultados_lotes,
        names=["lote", "linha"],
    ).reset_index(level="lote")

    colunas_resumo = [
        "lote",
        "ano",
        "status",
        "layout",
        "total_linhas",
        "total_participantes",
        "total_resultados",
        "staging_liberado_mb",
        "erro",
        "erro_limpeza",
    ]

    display(resumo_execucao[colunas_resumo])

    assert (resumo_execucao["status"] != "erro").all()
    assert resumo_execucao["erro"].isna().all()
    assert resumo_execucao["erro_limpeza"].isna().all()
else:
    resumo_execucao = None
    print("Não há resultados de execução nesta sessão.")


,lote,ano,status,layout,total_linhas,total_participantes,total_resultados,staging_liberado_mb,erro,erro_limpeza
linha,,,,,,,,,,
0,1998_2003,1998,processado,unico,288,NaN,NaN,60.144619,None,None
1,1998_2003,1999,processado,unico,6518,NaN,NaN,138.045591,None,None
2,1998_2003,2000,processado,unico,6219,NaN,NaN,165.594666,None,None
3,1998_2003,2001,processado,unico,43918,NaN,NaN,1026.466652,None,None
4,1998_2003,2002,processado,unico,36407,NaN,NaN,1014.886230,None,None
5,1998_2003,2003,processado,unico,44721,NaN,NaN,950.585272,None,None
0,2004_2008,2004,processado,unico,36351,NaN,NaN,806.656661,None,None
1,2004_2008,2005,processado,unico,72163,NaN,NaN,1662.940553,None,None
2,2004_2008,2006,processado,unico,104890,NaN,NaN,2080.403831,None,None


## 7. Auditoria final das 28 edições

A auditoria final chama novamente o pipeline com reutilização habilitada. Nenhum CSV é extraído quando todas as Silver existem. Cada Parquet é lido e validado.


In [33]:
EXECUTAR_AUDITORIA_FINAL = True

In [34]:
auditoria_final = None

if EXECUTAR_AUDITORIA_FINAL:
    auditoria_final = processar_varios_anos(
        anos=range(ANO_INICIAL, ANO_FINAL + 1),
        uf=UF_PROCESSAMENTO,
        catalogo=catalogo,
        reutilizar_silver=True,
        continuar_em_erro=False,
        limpar_staging_apos_sucesso=LIMPAR_STAGING_APOS_SUCESSO,
    )

    assert len(auditoria_final) == 28
    assert (auditoria_final["status"] == "reutilizado").all()
    assert auditoria_final["erro"].isna().all()
    assert auditoria_final["erro_limpeza"].isna().all()

    print("As 28 edições foram reutilizadas e validadas.")
else:
    print("Auditoria final não executada.")


[1/28] Processando 1998...
  reutilizado: 288 registros.
  staging: 0.00 MB liberados.
[2/28] Processando 1999...
  reutilizado: 6,518 registros.
  staging: 0.00 MB liberados.
[3/28] Processando 2000...
  reutilizado: 6,219 registros.
  staging: 0.00 MB liberados.
[4/28] Processando 2001...
  reutilizado: 43,918 registros.
  staging: 0.00 MB liberados.
[5/28] Processando 2002...
  reutilizado: 36,407 registros.
  staging: 0.00 MB liberados.
[6/28] Processando 2003...
  reutilizado: 44,721 registros.
  staging: 0.00 MB liberados.
[7/28] Processando 2004...
  reutilizado: 36,351 registros.
  staging: 0.00 MB liberados.
[8/28] Processando 2005...
  reutilizado: 72,163 registros.
  staging: 0.00 MB liberados.
[9/28] Processando 2006...
  reutilizado: 104,890 registros.
  staging: 0.00 MB liberados.
[10/28] Processando 2007...
  reutilizado: 101,145 registros.
  staging: 0.00 MB liberados.
[11/28] Processando 2008...
  reutilizado: 108,761 registros.
  staging: 0.00 MB liberados.
[12/28] Pr

## 8. Manifesto normalizado da Silver

O manifesto contém uma linha por Parquet: 26 linhas do layout único e quatro do layout separado.


In [35]:
def normalizar_manifesto(auditoria):
    registros = []

    for resultado in auditoria.to_dict(orient="records"):
        if resultado["layout"] == "unico":
            registros.append(
                {
                    "ano": int(resultado["ano"]),
                    "uf": resultado["uf"],
                    "layout": "unico",
                    "base": "microdados",
                    "identificador": "NU_INSCRICAO",
                    "total_linhas": int(resultado["total_linhas"]),
                    "arquivo": str(resultado["arquivo"]),
                    "tamanho_mb": float(resultado["tamanho_mb"]),
                    "status": resultado["status"],
                }
            )
            continue

        for base in ["participantes", "resultados"]:
            informacoes = resultado[base]
            registros.append(
                {
                    "ano": int(resultado["ano"]),
                    "uf": resultado["uf"],
                    "layout": "separado",
                    "base": base,
                    "identificador": informacoes["identificador"],
                    "total_linhas": int(informacoes["total_linhas"]),
                    "arquivo": str(informacoes["arquivo"]),
                    "tamanho_mb": float(informacoes["tamanho_mb"]),
                    "status": informacoes["status"],
                }
            )

    return (
        pd.DataFrame(registros)
        .sort_values(["ano", "base"])
        .reset_index(drop=True)
    )


manifesto_silver = None

if auditoria_final is not None:
    manifesto_silver = normalizar_manifesto(auditoria_final)
    display(manifesto_silver)
else:
    print("Execute a auditoria final antes de criar o manifesto.")


,ano,uf,layout,base,identificador,total_linhas,arquivo,tamanho_mb,status
0,1998,PA,unico,microdados,NU_INSCRICAO,288,/home/akel/PycharmProjects/ENEM/data/silver/mi...,0.015604,reutilizado
1,1999,PA,unico,microdados,NU_INSCRICAO,6518,/home/akel/PycharmProjects/ENEM/data/silver/mi...,0.206043,reutilizado
2,2000,PA,unico,microdados,NU_INSCRICAO,6219,/home/akel/PycharmProjects/ENEM/data/silver/mi...,0.199291,reutilizado
3,2001,PA,unico,microdados,NU_INSCRICAO,43918,/home/akel/PycharmProjects/ENEM/data/silver/mi...,1.156630,reutilizado
4,2002,PA,unico,microdados,NU_INSCRICAO,36407,/home/akel/PycharmProjects/ENEM/data/silver/mi...,0.974503,reutilizado
5,2003,PA,unico,microdados,NU_INSCRICAO,44721,/home/akel/PycharmProjects/ENEM/data/silver/mi...,1.101504,reutilizado
6,2004,PA,unico,microdados,NU_INSCRICAO,36351,/home/akel/PycharmProjects/ENEM/data/silver/mi...,0.852600,reutilizado
7,2005,PA,unico,microdados,NU_INSCRICAO,72163,/home/akel/PycharmProjects/ENEM/data/silver/mi...,2.004443,reutilizado
8,2006,PA,unico,microdados,NU_INSCRICAO,104890,/home/akel/PycharmProjects/ENEM/data/silver/mi...,3.110282,reutilizado
9,2007,PA,unico,microdados,NU_INSCRICAO,101145,/home/akel/PycharmProjects/ENEM/data/silver/mi...,2.842739,reutilizado


In [36]:
if manifesto_silver is not None:
    assert len(manifesto_silver) == 30
    assert manifesto_silver["ano"].nunique() == 28
    assert manifesto_silver["arquivo"].map(lambda item: Path(item).exists()).all()
    assert (manifesto_silver["total_linhas"] > 0).all()
    assert (manifesto_silver["tamanho_mb"] > 0).all()

    print("Manifesto validado: 30 Parquets e 28 edições.")


Manifesto validado: 30 Parquets e 28 edições.


In [37]:
arquivo_manifesto = METADATA_DIR / f"manifesto_silver_{UF_PROCESSAMENTO}.csv"
SALVAR_MANIFESTO = True
if SALVAR_MANIFESTO:
    if manifesto_silver is None:
        raise RuntimeError("Execute a auditoria final e crie o manifesto antes de salvá-lo.")

    arquivo_manifesto.parent.mkdir(parents=True, exist_ok=True)
    manifesto_silver.to_csv(
        arquivo_manifesto,
        sep=";",
        index=False,
        encoding="utf-8",
    )

    print(f"Manifesto salvo em: {arquivo_manifesto}")
else:
    print(f"Salvamento desativado. Destino previsto: {arquivo_manifesto}")


Manifesto salvo em: /home/akel/PycharmProjects/ENEM/data/metadata/manifesto_silver_PA.csv


## 9. Verificação do staging

Quando a limpeza está habilitada e todos os anos foram concluídos, não devem restar CSVs principais dentro de `data/staging/ano_AAAA`.


In [38]:
csvs_restantes_staging = list(STAGING_DIR.glob("ano_*/*.csv"))

print("CSVs restantes no staging:", len(csvs_restantes_staging))

if csvs_restantes_staging:
    display(pd.DataFrame({"arquivo": csvs_restantes_staging}))
elif EXECUTAR_PIPELINE or EXECUTAR_AUDITORIA_FINAL:
    print("Staging completamente limpo.")


CSVs restantes no staging: 0
Staging completamente limpo.


## Conclusão

Ao final de uma execução completa e validada, o projeto deve apresentar:

- 28 edições processadas;
- 30 Parquets na Silver;
- nenhuma duplicação ou UF indevida;
- participantes e resultados mantidos separadamente em 2024–2025;
- manifesto salvo em `data/metadata`;
- staging limpo, quando essa opção estiver habilitada.

A construção de tabelas analíticas e comparações longitudinais pertence à etapa Gold e deve respeitar a quebra metodológica introduzida em 2024.
